# Tutorial: Decorators

1. What are decorators, and how do they work?
2. Creating simple decorators
3. Creating decorators that modify inputs
4. Creating decorators that modify outputs
5. Multiple decorators
6. Decorators that take arguments
7. Improving our decorators with ... decorators!
8. (Maybe) decorating classes, decorating with classes

# DRY -- don't repeat yourself!



In [1]:
def a():
    return f'a!\n'

def b():
    return f'b!\n'

print(a())    
print(b())

a!

b!



In [2]:
# we need to print dashed lines above and below every printout

lines = '-' * 60 + '\n'

def a():
    return f'{lines}a!\n{lines}'

def b():
    return f'{lines}b!\n{lines}'

print(a())    
print(b())

------------------------------------------------------------
a!
------------------------------------------------------------

------------------------------------------------------------
b!
------------------------------------------------------------



In [3]:
# option 2: instead of calling the function directly,
# let's call it via another function which will insert the lines

lines = '-' * 60 + '\n'

def with_lines(func):
    return f'{lines}{func()}{lines}'

def a():
    return f'a!\n'

def b():
    return f'b!\n'

print(with_lines(a))
print(with_lines(b))

------------------------------------------------------------
a!
------------------------------------------------------------

------------------------------------------------------------
b!
------------------------------------------------------------



In [4]:
# option 3: The problem with the previous option is that
# it broke existing code!

# let's instead use a nested function -- a function defined
# inside of another function!

lines = '-' * 60 + '\n'

def with_lines(func):
    def wrapper():
        return f'{lines}{func()}{lines}'
    return wrapper

def a():
    return f'a!\n'
with_lines_a = with_lines(a)

def b():
    return f'b!\n'
with_lines_b = with_lines(b)

print(with_lines_a())
print(with_lines_b())

------------------------------------------------------------
a!
------------------------------------------------------------

------------------------------------------------------------
b!
------------------------------------------------------------



In [5]:
# option 4: Now let's avoid breaking existing code

lines = '-' * 60 + '\n'

def with_lines(func):
    def wrapper():
        return f'{lines}{func()}{lines}'
    return wrapper

def a():
    return f'a!\n'
a = with_lines(a)

def b():
    return f'b!\n'
b = with_lines(b)

print(a())
print(b())

------------------------------------------------------------
a!
------------------------------------------------------------

------------------------------------------------------------
b!
------------------------------------------------------------



In [6]:
# option 5: use decorator syntax

lines = '-' * 60 + '\n'

def with_lines(func):
    def wrapper():
        return f'{lines}{func()}{lines}'
    return wrapper

@with_lines    # this line is 100% equivalent to line 13
def a():
    return f'a!\n'
# a = with_lines(a)

@with_lines   # this is 100% equivalent to line 18
def b():
    return f'b!\n'
# b = with_lines(b)

print(a())
print(b())

------------------------------------------------------------
a!
------------------------------------------------------------

------------------------------------------------------------
b!
------------------------------------------------------------



# Summary of a decorator

1. A decorator is a function, which we mention with @
2. That decorator function takes an argument, a function (the "decorated" function)
3. The decorator returns a function (the inner function, often named "wrapper")
4. The returned wrapper then is assigned to the name that the decorated function originally had.

# Who cares?

A decorator allows me to hijack a function at definition time (when the outer, decorator function runs), and also at runtime (when the inner, wrapper function runs).

1. Replace a function if the permissions are wrong
2. Only allow a function to run under particular circumstances
3. Change/filter the inputs to the function
4. change/filter the outputs from the function
5. Log information about the function's running

Everything we do with decorators could be done by modifying the function itself. But that would violate the DRY rule. This allows us to apply functionality to a large number of functions without rewriting them.

In [9]:
# let's use this decorator in more places!

lines = '-' * 60 + '\n'

def with_lines(func):
    def wrapper(*args):
        return f'{lines}{func(*args)}{lines}'
    return wrapper

@with_lines    # this line is 100% equivalent to line 13
def a():
    return f'a!\n'

@with_lines   # this is 100% equivalent to line 18
def b():
    return f'b!\n'

print(a())
print(b())

@with_lines
def add(x, y):
    return x + y

@with_lines
def mul(x, y):
    return x * y

print(add(2, 5))    
print(add(10, 20))
print(mul(3, 7))
print(mul(10, 15))

------------------------------------------------------------
a!
------------------------------------------------------------

------------------------------------------------------------
b!
------------------------------------------------------------

------------------------------------------------------------
7------------------------------------------------------------

------------------------------------------------------------
30------------------------------------------------------------

------------------------------------------------------------
21------------------------------------------------------------

------------------------------------------------------------
150------------------------------------------------------------



# Exercise: Timing

1. Write a decorator, called `timefunc`, that checks how long it takes for a function to run. The function will run normally, with its usual arguments and returning its usual values, but we will keep track of how long it took to execute.
2. The execution time, along with the name of the function we were running, will be stored to a file called `timing.txt`.
3. You can get the current Unix time (i.e., number of seconds since Jan 1, 1970) with `time.time()`.
4. You can get the name of a function from its `__name__` attribute.
5. You can write two slow functions -- I usually write `slow_add` and `slow_mul`, and put in `time.sleep` in there, maybe even using a random number.

In [13]:
import time
import random

def timefunc(func):     # outer decorator function always takes a function
    def wrapper(*args): # wrapper always takes *args
        start_time = time.time()
        value = func(*args)
        total_time = time.time() - start_time

        with open('timing.txt', 'a') as f:
            f.write(f'{func.__name__}\t{total_time}\n')
        
        return value
    return wrapper

@timefunc
def slow_add(x, y):
    time.sleep(random.randint(0, 3))
    return x + y

@timefunc
def slow_mul(x, y):
    time.sleep(random.randint(0, 3))
    return x * y

print(slow_add(2, 5))    
print(slow_add(10, 20))
print(slow_mul(3, 7))
print(slow_mul(10, 15))

7
30
21
150


In [14]:
!cat timing.txt

slow_add	2.0050621032714844
slow_add	1.001225233078003
slow_mul	3.003403902053833
slow_mul	4.38690185546875e-05


# Caching

What if we set things up such that `wrapper` will check the arguments of the function that is being run. If we've never seen these arguments before, then we will run the function, storing the result in a cache. If we have seen the arguments before, we just return the result that we saw before.

This kind of caching is known as "memoization."



In [16]:
def memoize(func):
    cache = {}

    def wrapper(*args):
        if args not in cache:          # first time with these args?
            cache[args] = func(*args)  # store in our cache
        return cache[args]             # always works!
    return wrapper

@memoize
def slow_add(x, y):
    time.sleep(random.randint(0, 3))
    return x + y

@memoize
def slow_mul(x, y):
    time.sleep(random.randint(0, 3))
    return x * y

print(slow_add(2, 5))    
print(slow_add(10, 20))
print(slow_mul(3, 7))
print(slow_mul(10, 15))
print(slow_add(2, 5))    
print(slow_add(10, 20))
print(slow_mul(3, 7))
print(slow_mul(10, 15))

7
30
21
150
7
30
21
150


# Exercise: `once_per_minute`

Write a decorator, `once_per_minute`, that when applied to a function, raises an exception (`CalledTooSoon`) if the function is invoked more than once in a 60-second period.

In [22]:
class CalledTooSoonError(Exception):
    pass

def once_per_minute(func):
    last_called_at = 0

    def wrapper(*args):
        nonlocal last_called_at
        current_time = time.time()

        remaining_time = current_time - last_called_at
        if remaining_time < 60:
            raise CalledTooSoonError(f'Too soon; wait {remaining_time} more')

        last_called_at = current_time
        return func(*args)
    return wrapper

@once_per_minute
def slow_add(x, y):
    time.sleep(random.randint(0, 3))
    return x + y

@once_per_minute
def slow_mul(x, y):
    time.sleep(random.randint(0, 3))
    return x * y

print(slow_add(2, 5))    
print(slow_mul(3, 7))
print(slow_add(10, 20))
print(slow_mul(10, 15))


7
21


CalledTooSoonError: Too soon; wait 5.009716987609863 more

In [21]:
x = 100    # this takes the value (100) and assigns it to the variable

In [ ]:
d = {}    # this assigns a dict to d
d['a'] = 100   # this actually is rewritten to

# d.__setitem__('a', 100)